# 🇲🇦 RAG System — Documents Juridiques Marocains (مدونة الأسرة)

**Pipeline:**  
`PDF Loading → Text Extraction → Chunking → Embeddings → Vector Store → Retrieval → Generation`

Ce système permet d'interagir en **arabe** avec les textes de loi marocains via un modèle de langage augmenté par la recherche (RAG).

## 1. Installation des dépendances

In [14]:
!pip install langchain langchain-community langchain-google-genai langchain-groq langchain-chroma \
    langchain-text-splitters pypdf chromadb google-generativeai python-dotenv \
    unstructured[pdf] unstructured-inference pdf2image pdfminer.six pillow_heif -q


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
for pkg in ["langchain", "langchain-community", "langchain-google-genai", "langchain-groq", "langchain-chroma",
            "langchain-text-splitters", "pypdf", "chromadb", "google-generativeai", "python-dotenv",
            "unstructured", "unstructured-inference", "pdf2image", "pdfminer.six", "pillow_heif"]:
    print(f"{pkg}:")
    !pip show {pkg}
    print()

## 2. Imports & Configuration

In [15]:
import os
import glob
from pathlib import Path

# Legacy loader
from langchain_community.document_loaders import PyPDFLoader
# Unstructured loader
from langchain_community.document_loaders import UnstructuredPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

from dotenv import load_dotenv

# ── Load .env (create a .env file with GOOGLE_API_KEY=your_key) ──
load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
if not GOOGLE_API_KEY:
    GOOGLE_API_KEY = input("🔑 Entrez votre Google API Key (pour embeddings): ")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    GROQ_API_KEY = input("🔑 Entrez votre Groq API Key (pour LLM — https://console.groq.com): ")
    os.environ["GROQ_API_KEY"] = GROQ_API_KEY

print("✅ Configuration chargée avec succès.")

✅ Configuration chargée avec succès.


## 3. Fonctions du Pipeline RAG

In [16]:
# ═══════════════════════════════════════════════════════════════════
# 3.1  get_pdfs  —  Discover all PDF files in a folder
# ═══════════════════════════════════════════════════════════════════

def get_pdfs(folder_path: str) -> list[str]:
    """
    Scan *folder_path* and return a sorted list of absolute paths
    to every .pdf file found (non-recursive by default).
    """
    pattern = os.path.join(folder_path, "*.pdf")
    pdf_paths = sorted(glob.glob(pattern))
    print(f"📂 {len(pdf_paths)} PDF(s) trouvé(s) dans '{folder_path}'")
    for p in pdf_paths:
        print(f"   • {os.path.basename(p)}")
    return pdf_paths

In [27]:
# ═══════════════════════════════════════════════════════════════════
# 3.2  get_pdfs_text  —  Extract text (pages/elements) from PDFs
# ═══════════════════════════════════════════════════════════════════

from langchain_core.documents import Document

def get_pdfs_text(pdf_paths: list[str], use_unstructured: bool = False) -> list[Document]:
    """
    Load PDFs and return a list of Documents.
    
    Args:
        pdf_paths: List of file paths
        use_unstructured: If True, uses UnstructuredPDFLoader for advanced loading/chunking.
                          (Ensure 'unstructured' packages are installed)
    """
    all_docs: list[Document] = []
    
    for path in pdf_paths:
        if use_unstructured:
            print(f"   ⚙️ Processing {os.path.basename(path)} with Unstructured... (patience)")
            try:
                # 'hi_res' is better for Arabic/complex layouts but requires system dependencies (tesseract).
                # 'fast' is text-only. If it fails, fallback to PyPDF.
                loader = UnstructuredPDFLoader(
                    path, 
                    mode="elements", 
                    strategy="fast",
                    languages=["ara"],  # Specify Arabic language
                )
                data = loader.load()
                
                # Check if extraction actually worked
                if len(data) == 0:
                     print(f"   ⚠️ 'Fast' strategy returned 0 elements. Trying PyPDFLoader as fallback.")
                     raise ValueError("Zero elements found")

                print(f"   📄 {os.path.basename(path)}: {len(data)} elements (chunks)")
                all_docs.extend(data)

            except Exception as e:
                print(f"   ⚠️ Unstructured issue: {e}")
                print("   🔄 Fallback to PyPDFLoader (Standard extraction)...")
                
                try:
                    loader = PyPDFLoader(path)
                    pages = loader.load()
                    all_docs.extend(pages)
                    print(f"   📄 {os.path.basename(path)}: {len(pages)} page(s) (via PyPDFLoader)")
                except Exception as ex:
                    print(f"   ❌ Échec total pour ce fichier: {ex}")

        else:
            loader = PyPDFLoader(path)
            pages = loader.load()          
            all_docs.extend(pages)
            print(f"   📄 {os.path.basename(path)}: {len(pages)} page(s)")
            
    print(f"\n✅ Total: {len(all_docs)} documents/elements extraits")
    return all_docs

In [37]:
# ═══════════════════════════════════════════════════════════════════
# 3.3  chunk_documents  —  Split documents into smaller chunks
# ═══════════════════════════════════════════════════════════════════

from langchain_text_splitters import RecursiveCharacterTextSplitter

def chunk_documents(
    documents: list[Document],
    chunk_size: int = 1000,
    chunk_overlap: int = 200,
    use_unstructured_chunking: bool = False
) -> list[Document]:
    """
    Split documents into overlapping chunks suitable for embedding.
    """
    
    # Séparateurs optimisés pour l'arabe et les textes juridiques
    # Priorité : Double saut de ligne (paragraphe) > Saut de ligne > Point > Virgule > Espace
    separators = [
        "\n\n",   # Paragraphes
        "\n",     # Lignes
        ".",      # Phrases
        "؛",      # Point-virgule arabe
        "،",      # Virgule arabe
        " ",      # Mots
        ""        # Caractères
    ]

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=separators,
        length_function=len,
        keep_separator=True # Garde la ponctuation dans le chunk
    )
    
    chunks = splitter.split_documents(documents)
    
    # Nettoyage : retirer les chunks trop petits (ex: numéros de page isolés)
    cleaned_chunks = []
    for c in chunks:
        content = c.page_content.strip()
        if len(content) > 30: # Un chunk utile fait au moins quelques mots
            cleaned_chunks.append(c)

    print(f"✅ {len(documents)} document(s) → {len(cleaned_chunks)} chunks raffinés "
          f"(size={chunk_size}, overlap={chunk_overlap})")
    return cleaned_chunks

In [38]:
# ═══════════════════════════════════════════════════════════════════
# 3.4  get_embeddings  —  Build the embedding model
# ═══════════════════════════════════════════════════════════════════

def get_embeddings(model_name: str = "models/gemini-embedding-001") -> GoogleGenerativeAIEmbeddings:
    """
    Return a Google Generative AI embedding model.
    This model supports Arabic text natively.
    """
    embeddings = GoogleGenerativeAIEmbeddings(model=model_name)
    print(f"✅ Modèle d'embeddings chargé: {model_name}")
    return embeddings

In [39]:
# ═══════════════════════════════════════════════════════════════════
# 3.5  create_vector_store  —  Index chunks in ChromaDB
# ═══════════════════════════════════════════════════════════════════

import time

CHROMA_PERSIST_DIR = "./chroma_db"

def create_vector_store(
    chunks: list[Document],
    embeddings: GoogleGenerativeAIEmbeddings,
    persist_directory: str = CHROMA_PERSIST_DIR,
    batch_size: int = 50,
    delay: float = 62.0,
) -> Chroma:
    """
    Create a Chroma vector store from document chunks.
    Embeds in batches with delay to respect free-tier rate limits
    (100 requests/min for Gemini embedding).
    """
    # Create empty store first
    vectorstore = Chroma(
        embedding_function=embeddings,
        persist_directory=persist_directory,
    )

    total = len(chunks)
    for i in range(0, total, batch_size):
        batch = chunks[i : i + batch_size]
        texts = [doc.page_content for doc in batch]
        metadatas = [doc.metadata for doc in batch]
        vectorstore.add_texts(texts=texts, metadatas=metadatas)
        done = min(i + batch_size, total)
        print(f"   ⏳ {done}/{total} chunks indexés...")

        # Wait between batches to avoid rate-limit (skip after last batch)
        if done < total:
            print(f"   💤 Pause de {delay}s pour respecter le rate-limit...")
            time.sleep(delay)

    print(f"✅ Vector store créé avec {total} chunks "
          f"(persisté dans '{persist_directory}')")
    return vectorstore


def load_vector_store(
    embeddings: GoogleGenerativeAIEmbeddings,
    persist_directory: str = CHROMA_PERSIST_DIR,
) -> Chroma:
    """
    Load an existing Chroma vector store from disk.
    """
    vectorstore = Chroma(
        persist_directory=persist_directory,
        embedding_function=embeddings,
    )
    print(f"✅ Vector store chargé depuis '{persist_directory}'")
    return vectorstore

In [40]:
# ═══════════════════════════════════════════════════════════════════
# 3.6  get_retriever  —  Build the retriever from the vector store
# ═══════════════════════════════════════════════════════════════════

def get_retriever(vectorstore: Chroma, k: int = 10):
    """
    Wrap the vector store in a retriever that returns the top-k
    most similar chunks for a given query.
    """
    retriever = vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": k},
    )
    print(f"✅ Retriever configuré (top-{k} chunks)")
    return retriever

In [41]:
# ═══════════════════════════════════════════════════════════════════
# 3.7  get_llm  —  Initialise the generative model
# ═══════════════════════════════════════════════════════════════════

def get_llm(
    model_name: str = "meta-llama/llama-4-scout-17b-16e-instruct",
    temperature: float = 0.3,
) -> ChatGroq:
    """
    Return a Groq-hosted LLM (Llama 4 Scout).
    Free tier: 30 req/min, 14,400 req/day.
    Good Arabic support.
    """
    llm = ChatGroq(
        model=model_name,
        temperature=temperature,
    )
    print(f"✅ LLM chargé (Groq): {model_name} (temperature={temperature})")
    return llm

In [42]:
# ═══════════════════════════════════════════════════════════════════
# 3.8  build_rag_chain  —  Assemble Retrieval + Generation
# ═══════════════════════════════════════════════════════════════════

def format_docs(docs):
    """Concatenate retrieved document contents into a single string."""
    return "\n\n".join(doc.page_content for doc in docs)


def build_rag_chain(llm, retriever):
    """
    Build a RAG chain using LCEL (LangChain Expression Language).
    The prompt instructs the LLM to answer strictly from context,
    cite article numbers, and reply in Arabic.
    """

    prompt_template = """أنت مساعد قانوني متخصص في القانون المغربي، وخاصة مدونة الأسرة.
أجب على السؤال بناءً فقط على السياق المقدم أدناه.
إذا لم تجد الإجابة في السياق، قل: "لم أجد معلومات كافية في النص للإجابة على هذا السؤال."
استشهد بأرقام المواد عند الإمكان.

السياق:
{context}

السؤال: {question}

الجواب:"""

    PROMPT = PromptTemplate(
        template=prompt_template,
        input_variables=["context", "question"],
    )

    # LCEL chain: retrieve → format → prompt → LLM → parse
    chain = (
        {
            "context": retriever | format_docs,
            "question": RunnablePassthrough(),
        }
        | PROMPT
        | llm
        | StrOutputParser()
    )
    print("✅ RAG chain prête (LCEL).")
    return chain, retriever

In [43]:
# ═══════════════════════════════════════════════════════════════════
# 3.9  ask  —  Query the RAG system
# ═══════════════════════════════════════════════════════════════════

def ask(chain, retriever, question: str, max_retries: int = 3) -> dict:
    """
    Send a question to the RAG chain.
    Retries automatically on rate-limit errors.
    Returns a dict with 'result' (answer text) and 'source_documents'.
    """
    for attempt in range(max_retries):
        try:
            answer = chain.invoke(question)
            source_docs = retriever.invoke(question)
            return {"result": answer, "source_documents": source_docs}
        except Exception as e:
            if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
                wait = 35 * (attempt + 1)
                print(f"   ⚠️ Rate-limit atteint. Nouvelle tentative dans {wait}s... ({attempt+1}/{max_retries})")
                time.sleep(wait)
            else:
                raise
    raise RuntimeError("Échec après plusieurs tentatives (rate-limit).")


def ask_llm_only(llm, question: str) -> str:
    """
    Ask the LLM directly WITHOUT retrieval context.
    Used for comparison with RAG answers.
    """
    prompt = f"""أنت مساعد قانوني متخصص في القانون المغربي.
أجب على السؤال التالي بالعربية.

السؤال: {question}

الجواب:"""
    response = llm.invoke(prompt)
    return response.content


def display_answer(response: dict):
    """Pretty-print the answer and source references."""
    print("=" * 70)
    print("📝 الجواب:")
    print("=" * 70)
    print(response["result"])
    print("\n" + "-" * 70)
    print("📚 المصادر:")
    print("-" * 70)
    for i, doc in enumerate(response["source_documents"], 1):
        source = os.path.basename(doc.metadata.get("source", "—"))
        page = doc.metadata.get("page", "?")
        print(f"  [{i}] {source}  —  الصفحة {page}")

        # Show a short preview of the chunk
        preview = doc.page_content[:200].replace("\n", " ")
        print(f"      {preview}…\n")


def compare_rag_vs_llm(chain, retriever, llm, question: str):
    """
    Display side-by-side comparison: RAG answer vs. LLM-only answer.
    """
    print("═" * 70)
    print(f"❓ السؤال: {question}")
    print("═" * 70)

    # ── RAG Answer ──
    print("\n🟢 الجواب باستخدام RAG (مع السياق من الوثائق):")
    print("-" * 70)
    rag_response = ask(chain, retriever, question)
    print(rag_response["result"])
    print("\n📚 المصادر:")
    for i, doc in enumerate(rag_response["source_documents"], 1):
        source = os.path.basename(doc.metadata.get("source", "—"))
        page = doc.metadata.get("page", "?")
        print(f"  [{i}] {source} — الصفحة {page}")

    # ── LLM-only Answer ──
    print("\n\n🟡 الجواب بدون RAG (LLM فقط — معرفة عامة):")
    print("-" * 70)
    llm_answer = ask_llm_only(llm, question)
    print(llm_answer)

    print("\n" + "═" * 70)
    print("🔍 خلاصة: الجواب RAG مبني على نص القانون الفعلي مع ذكر المصادر،")
    print("  بينما الجواب LLM فقط يعتمد على المعرفة العامة للنموذج.")
    print("═" * 70)

## 4. Exécution du Pipeline

In [44]:
# ── 4.1  Load & Index PDFs (run once, then skip) ─────────────────

PDF_FOLDER = "./data"

# Step 1: Discover PDFs
pdf_paths = get_pdfs(PDF_FOLDER)

📂 1 PDF(s) trouvé(s) dans './data'
   • مدونة الأسرة-adala.pdf


In [45]:
# Step 2: Extract text 
# (Setting use_unstructured=True enables semantic chunking via partition_pdf)
documents = get_pdfs_text(pdf_paths, use_unstructured=True)

   ⚙️ Processing مدونة الأسرة-adala.pdf with Unstructured... (patience)
   ⚠️ 'Fast' strategy returned 0 elements. Trying PyPDFLoader as fallback.
   ⚠️ Unstructured issue: Zero elements found
   🔄 Fallback to PyPDFLoader (Standard extraction)...
   📄 مدونة الأسرة-adala.pdf: 107 page(s) (via PyPDFLoader)

✅ Total: 107 documents/elements extraits


In [46]:
documents

[Document(metadata={'producer': 'Microsoft® Word\xa0LTSC', 'creator': 'Microsoft® Word\xa0LTSC', 'creationdate': '2024-07-22T12:13:38+01:00', 'author': 'Douk', 'moddate': '2024-07-22T12:13:38+01:00', 'source': './data\\مدونة الأسرة-adala.pdf', 'total_pages': 107, 'page': 0, 'page_label': '1'}, page_content='- 1  - \n \n \n \n \n \n \n \n \n \n \nمدونة الأسرة \nصيغة محينة بتاريخ 29 يوليو 2021'),
 Document(metadata={'producer': 'Microsoft® Word\xa0LTSC', 'creator': 'Microsoft® Word\xa0LTSC', 'creationdate': '2024-07-22T12:13:38+01:00', 'author': 'Douk', 'moddate': '2024-07-22T12:13:38+01:00', 'source': './data\\مدونة الأسرة-adala.pdf', 'total_pages': 107, 'page': 1, 'page_label': '2'}, page_content='- 2  - \nالقانون رقم70.03 بمثابة مدونة الأسرة1 \nكما تم تعديله ب:  \n- القانون رقم  65.21  القاضي بتغيير وتتميم المادة15  من القانون رقم70.03  بمثابة\nمدونة الأسرة، الصادر بتنفيذه الظهير الشريف رقم  1.21.73  بتاريخ3  ذي الحجة\n1442(14  يوليو2021)، الجريدة الرسمية عدد7008  بتاريخ18  ذو الحجة14

In [47]:
# Step 3: Chunk
chunks = chunk_documents(documents, chunk_size=1000, chunk_overlap=200)

✅ 107 document(s) → 148 chunks raffinés (size=1000, overlap=200)


In [48]:
chunks

[Document(metadata={'producer': 'Microsoft® Word\xa0LTSC', 'creator': 'Microsoft® Word\xa0LTSC', 'creationdate': '2024-07-22T12:13:38+01:00', 'author': 'Douk', 'moddate': '2024-07-22T12:13:38+01:00', 'source': './data\\مدونة الأسرة-adala.pdf', 'total_pages': 107, 'page': 0, 'page_label': '1'}, page_content='- 1  - \n \n \n \n \n \n \n \n \n \n \nمدونة الأسرة \nصيغة محينة بتاريخ 29 يوليو 2021'),
 Document(metadata={'producer': 'Microsoft® Word\xa0LTSC', 'creator': 'Microsoft® Word\xa0LTSC', 'creationdate': '2024-07-22T12:13:38+01:00', 'author': 'Douk', 'moddate': '2024-07-22T12:13:38+01:00', 'source': './data\\مدونة الأسرة-adala.pdf', 'total_pages': 107, 'page': 1, 'page_label': '2'}, page_content='- 2  - \nالقانون رقم70.03 بمثابة مدونة الأسرة1 \nكما تم تعديله ب:  \n- القانون رقم  65.21  القاضي بتغيير وتتميم المادة15  من القانون رقم70.03  بمثابة\nمدونة الأسرة، الصادر بتنفيذه الظهير الشريف رقم  1.21.73  بتاريخ3  ذي الحجة\n1442(14  يوليو2021)، الجريدة الرسمية عدد7008  بتاريخ18  ذو الحجة14

In [49]:
# Step 4: Embeddings
embeddings = get_embeddings()

✅ Modèle d'embeddings chargé: models/gemini-embedding-001


In [94]:
# 🔍 List available embedding models from your API key
import google.genai as genai

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
for m in client.models.list():
    if "embed" in m.name.lower():
        print(m.name, "→", m.supported_actions)

models/gemini-embedding-001 → ['embedContent', 'countTextTokens', 'countTokens', 'asyncBatchEmbedContent']


In [92]:
# Step 5: Create Vector Store
vectorstore = create_vector_store(chunks, embeddings)

   ⏳ 50/148 chunks indexés...
   💤 Pause de 62.0s pour respecter le rate-limit...
   ⏳ 100/148 chunks indexés...
   💤 Pause de 62.0s pour respecter le rate-limit...
   ⏳ 148/148 chunks indexés...
✅ Vector store créé avec 148 chunks (persisté dans './chroma_db')


In [19]:
# ── 4.2  Build the RAG Chain ──────────────────────────────────────

# Charger le vector store existant depuis le disque
embeddings  = get_embeddings()
vectorstore = load_vector_store(embeddings)

retriever = get_retriever(vectorstore, k=5)
llm       = get_llm()  # Groq / Llama 4 Scout
rag_chain, retriever = build_rag_chain(llm, retriever)

✅ Modèle d'embeddings chargé: models/gemini-embedding-001
✅ Vector store chargé depuis './chroma_db'
✅ Retriever configuré (top-5 chunks)
✅ LLM chargé (Groq): meta-llama/llama-4-scout-17b-16e-instruct (temperature=0.3)
✅ RAG chain prête (LCEL).


## 5. Interrogation du système

In [20]:
# ── 5.1  Exemple — Question sur مدونة الأسرة ─────────────────────

question = "ما هي شروط الزواج في مدونة الأسرة المغربية؟"

response = ask(rag_chain, retriever, question)
display_answer(response)

📝 الجواب:
المادة 13 من مدونة الأسرة المغربية تنص على أن عقد الزواج يجب أن تتوفر فيه الشروط الآتية:
1 - أهلية الزوج والزوجة؛ 
2 - عدم الاتفاق على إسقاط الصداق؛ 
3 - ولي الزواج عند الاقتضاء؛ 
4 - سماع العدلين التصريح بالإيجاب والقبول من الزوجين وتوثيقه؛  
5 - انتفاء الموانع الشرعية.

----------------------------------------------------------------------
📚 المصادر:
----------------------------------------------------------------------
  [1] مدونة الأسرة-adala.pdf  —  الصفحة 12
      - 13  -  المادة12  تطبق على عقد الزواج المشوب بإكراه أو تدليس الأحكام المنصوص عليها في المادتين   63 و66 بعده  المادة 13  يجب أن تتوفر في عقد الزواج الشروط الآتية  1 - أهلية الزوج والزوجة؛  2 - عدم ال…

  [2] مدونة الأسرة-adala.pdf  —  الصفحة 12
      - 13  -  المادة12  تطبق على عقد الزواج المشوب بإكراه أو تدليس الأحكام المنصوص عليها في المادتين   63 و66 بعده  المادة 13  يجب أن تتوفر في عقد الزواج الشروط الآتية  1 - أهلية الزوج والزوجة؛  2 - عدم ال…

  [3] مدونة الأسرة-adala.pdf  —  الصفحة 12
      - 13  -  ال

In [21]:
# ── 5.2  Comparaison : RAG vs LLM seul ────────────────────────────

# Question difficile et très spécifique pour montrer la différence
question = "ما هي الشروط القانونية للتعدد (تعدد الزوجات) في مدونة الأسرة المغربية، وما هي الحالات التي يمنع فيها التعدد؟"

compare_rag_vs_llm(rag_chain, retriever, llm, question)

══════════════════════════════════════════════════════════════════════
❓ السؤال: ما هي الشروط القانونية للتعدد (تعدد الزوجات) في مدونة الأسرة المغربية، وما هي الحالات التي يمنع فيها التعدد؟
══════════════════════════════════════════════════════════════════════

🟢 الجواب باستخدام RAG (مع السياق من الوثائق):
----------------------------------------------------------------------
طبقا للفصل 41 من مدونة الأسرة المغربية، لا تأذن المحكمة بالتعدد إذا لم يثبت لها المبرر الموضوعي الاستثنائي، أو إذا لم تكن لطالب التعدد الموارد الكافية لإعالة الأسرتين وضمان جميع الحقوق من نفقة وإسكان ومساواة في جميع أوجه الحياة. 

بالإضافة إلى ذلك، يجب على الزوج الذي يرغب في التعدد تقديم طلب إلى المحكمة، يتضمن بيان الأسباب الموضوعية الاستثنائية المبررة للتعدد، ومرفقا بإقرار عن وضعه المادي، وذلك وفقا للفصل 42.

كما أن المادة 44 تشترط أن يكون التعدد مبنيا على مبرر موضوعي استثنائي وتوفر شروطه الشرعية، مع تقييده بشروط لفائدة المتزوج عليها وأطفالهما.

بالنسبة للحالات التي يمنع فيها التعدد، فهي:

1. عدم وجود مبرر موضوعي

In [ ]:
# ── 5.2  Mode interactif — Posez vos questions en boucle ──────────

print("💬 Mode interactif — tapez 'exit' pour quitter\n")

while True:
    q = input("❓ سؤالك: ")
    if q.strip().lower() in ("exit", "quit", "خروج", ""):
        print("👋 Au revoir!")
        break
    resp = ask(rag_chain, retriever, q)
    display_answer(resp)